# Semantic Search + LLM untuk FAQ E-commerce

Alur end-to-end: **ingest → embed → store → retrieve → generate**

| Tahap | Teknologi |
|---|---|
| Embedding | Gemini Embeddings API (`gemini-embedding-001`) |
| Vector database | FAISS (`IndexFlatIP`, cosine similarity) |
| LLM | Gemini API (`gemini-3.1-flash`) |

> Sebelum menjalankan: simpan CSV knowledge base di `../data/ecommerce_faq_knowledge_base.csv` dan set environment variable `GEMINI_API_KEY`.

## 1. Import Library & Load Knowledge Base

In [ ]:
from dotenv import load_dotenv
import os
from google import genai  # Menggunakan library Gemini API terbaru
from pathlib import Path
from google.genai import types

PROJECT_ROOT = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "data" / "ecommerce_faq_knowledge_base.csv").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "CSV tidak ditemukan. Pastikan data/ecommerce_faq_knowledge_base.csv "
        "ada di root project."
    )
DATA_PATH = PROJECT_ROOT / "data" / "ecommerce_faq_knowledge_base.csv"
load_dotenv(PROJECT_ROOT / ".env")

API_KEY = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
if not API_KEY:
    raise RuntimeError("GEMINI_API_KEY belum diatur. Isi file .env project dengan GEMINI_API_KEY=...")

client = genai.Client(api_key=API_KEY)

EMBED_MODEL = "gemini-embedding-001"   # sesuaikan jika starter notebook memakai model lain
LLM_MODEL = "gemini-3.1-flash"         # sesuaikan jika starter notebook memakai model lain
EMBED_DIM = 768                        # dimensi output embedding (di-normalisasi manual)
TOP_K = 3

## 2. Menyiapkan Knowledge Base
Data asli tidak diubah: semua transformasi hanya terjadi di memori.

In [10]:
df = pd.read_csv(DATA_PATH)

print("Jumlah entri :", len(df))
print("Kolom        :", df.columns.tolist())
print("Missing value:\n", df.isna().sum())
print("Duplikat     :", df.duplicated().sum())
df.head()

Jumlah entri : 59
Kolom        : ['faq_id', 'category', 'question', 'answer']
Missing value:
 faq_id      0
category    0
question    0
answer      0
dtype: int64
Duplikat     : 0


,faq_id,category,question,answer
0,1,Pemesanan,Bagaimana cara melakukan pemesanan di platform...,"Pilih produk yang diinginkan, klik 'Tambah ke ..."
1,2,Pemesanan,Apakah saya bisa memesan tanpa membuat akun?,Saat ini pemesanan hanya bisa dilakukan setela...
2,3,Pemesanan,Bagaimana cara membatalkan pesanan yang sudah ...,Pesanan dapat dibatalkan melalui halaman 'Pesa...
3,4,Pemesanan,Berapa lama waktu yang dibutuhkan sebelum pesa...,Penjual biasanya memproses pesanan dalam waktu...
4,5,Pemesanan,Apakah saya bisa mengubah alamat pengiriman se...,Perubahan alamat hanya bisa dilakukan sebelum ...


## 2. Menyiapkan Unit Retrievable

Setiap entri FAQ (question + answer) diperlakukan sebagai satu unit yang akan di-embed dan disimpan ke vector database. Gabungkan question dan answer menjadi satu teks representatif per entri.

In [27]:
df = pd.read_csv(DATA_PATH)

print("Jumlah entri :", len(df))
print("Kolom        :", df.columns.tolist())
print("Missing value:\n", df.isna().sum())
print("Duplikat     :", df.duplicated().sum())
df.head()

Jumlah entri : 59
Kolom        : ['faq_id', 'category', 'question', 'answer']
Missing value:
 faq_id      0
category    0
question    0
answer      0
dtype: int64
Duplikat     : 0


,faq_id,category,question,answer
0,1,Pemesanan,Bagaimana cara melakukan pemesanan di platform...,"Pilih produk yang diinginkan, klik 'Tambah ke ..."
1,2,Pemesanan,Apakah saya bisa memesan tanpa membuat akun?,Saat ini pemesanan hanya bisa dilakukan setela...
2,3,Pemesanan,Bagaimana cara membatalkan pesanan yang sudah ...,Pesanan dapat dibatalkan melalui halaman 'Pesa...
3,4,Pemesanan,Berapa lama waktu yang dibutuhkan sebelum pesa...,Penjual biasanya memproses pesanan dalam waktu...
4,5,Pemesanan,Apakah saya bisa mengubah alamat pengiriman se...,Perubahan alamat hanya bisa dilakukan sebelum ...


In [11]:
# Deteksi nama kolom secara fleksibel (sesuaikan manual jika nama kolom berbeda)
def find_col(candidates):
    for c in df.columns:
        if any(k in c.lower() for k in candidates):
            return c
    return None

Q_COL = find_col(["question", "pertanyaan"])
A_COL = find_col(["answer", "jawaban"])
CAT_COL = find_col(["category", "kategori"])
print("Kolom question:", Q_COL, "| answer:", A_COL, "| category:", CAT_COL)
assert Q_COL and A_COL, "Kolom question/answer tidak ditemukan. Set Q_COL dan A_COL secara manual."

if CAT_COL:
    print(df[CAT_COL].value_counts())

Kolom question: question | answer: answer | category: category
category
Pemesanan           9
Pengiriman          9
Pembayaran          8
Pengembalian        8
Akun                7
Produk              7
Promo               6
Customer Service    5
Name: count, dtype: int64


In [12]:
# Gabungkan category + question + answer menjadi satu unit teks yang di-retrieve
def build_unit(row):
    parts = []
    if CAT_COL:
        parts.append(f"Kategori: {row[CAT_COL]}")
    parts.append(f"Pertanyaan: {row[Q_COL]}")
    parts.append(f"Jawaban: {row[A_COL]}")
    return "\n".join(parts)

df["retrieval_text"] = df.apply(build_unit, axis=1)
print(df["retrieval_text"].iloc[0])

Kategori: Pemesanan
Pertanyaan: Bagaimana cara melakukan pemesanan di platform ini?
Jawaban: Pilih produk yang diinginkan, klik 'Tambah ke Keranjang', lalu lanjutkan ke halaman checkout untuk mengisi alamat pengiriman dan metode pembayaran.


In [13]:
df['content'] = 'Q: ' + df['question'] + ' A: ' + df['answer']
documents = df['content'].tolist()
print('Contoh unit retrievable:')
print(documents[0])

Contoh unit retrievable:
Q: Bagaimana cara melakukan pemesanan di platform ini? A: Pilih produk yang diinginkan, klik 'Tambah ke Keranjang', lalu lanjutkan ke halaman checkout untuk mengisi alamat pengiriman dan metode pembayaran.


## 3. Membuat Embeddings

Ubah setiap unit retrievable menjadi vektor menggunakan Gemini Embeddings API.
Dokumen memakai `RETRIEVAL_DOCUMENT`, query memakai `RETRIEVAL_QUERY`. Vektor dinormalisasi L2 agar inner product = cosine similarity.

In [14]:
def with_retry(fn, retries=4, wait=5):
    for attempt in range(retries):
        try:
            return fn()
        except Exception as e:
            if attempt == retries - 1:
                raise
            print(f"Error ({e}); mencoba lagi dalam {wait}s...")
            time.sleep(wait)
            wait *= 2


def embed_texts(texts, task_type, batch_size=50):
    vectors = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        resp = with_retry(lambda: client.models.embed_content(
            model=EMBED_MODEL,
            contents=batch,
            config=types.EmbedContentConfig(task_type=task_type, output_dimensionality=EMBED_DIM),
        ))
        vectors.extend(e.values for e in resp.embeddings)
    arr = np.array(vectors, dtype="float32")
    faiss.normalize_L2(arr)
    return arr


doc_embeddings = embed_texts(df["retrieval_text"].tolist(), "RETRIEVAL_DOCUMENT")
print("Shape embedding:", doc_embeddings.shape)
assert doc_embeddings.shape == (len(df), EMBED_DIM)

Shape embedding: (59, 768)


In [17]:
# Inisialisasi client Gemini yang benar
api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")

client = genai.Client(api_key=api_key)

def get_embedding(text):
    # Menggunakan model 'gemini-embedding-2' yang terbukti didukung oleh API key Anda
    result = client.models.embed_content(
        model='gemini-embedding-2',
        contents=text
    )
    return result.embeddings[0].values

# Lakukan embedding untuk seluruh dokumen
try:
    embeddings = [get_embedding(doc) for doc in documents]
    embeddings = np.array(embeddings).astype('float32')
    print('Shape embeddings:', embeddings.shape)
except Exception as e:
    print("Error saat membuat embedding:", e)

Shape embeddings: (59, 3072)


## 4. Menyimpan Embeddings ke Vector Database (FAISS)

In [19]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print('Jumlah vektor tersimpan di index:', index.ntotal)

Jumlah vektor tersimpan di index: 59


## 5. Mengimplementasikan Semantic Search

Fungsi untuk mengambil beberapa entri FAQ paling relevan berdasarkan query pengguna.

In [20]:
def semantic_search(query, top_k=3):
    query_embedding = np.array([get_embedding(query)]).astype('float32')
    distances, indices = index.search(query_embedding, top_k)

    results = []
    for rank, idx in enumerate(indices[0]):
        results.append({
            'rank': rank + 1,
            'question': df.iloc[idx]['question'],
            'answer': df.iloc[idx]['answer'],
            'category': df.iloc[idx]['category'],
            'distance': distances[0][rank]
        })
    return results

# Contoh percobaan
test_results = semantic_search('kenapa paket saya belum sampai?')
for r in test_results:
    print(r)

{'rank': 1, 'question': 'Bagaimana jika paket saya belum sampai melebihi estimasi?', 'answer': 'Jika paket melebihi estimasi pengiriman lebih dari 3 hari, silakan hubungi customer service dengan menyertakan nomor resi untuk ditelusuri lebih lanjut.', 'category': 'Pengiriman', 'distance': np.float32(0.6173067)}
{'rank': 2, 'question': 'Kenapa status pengiriman saya tidak berubah selama beberapa hari?', 'answer': 'Status yang tidak berubah bisa disebabkan keterlambatan update sistem kurir. Disarankan menunggu 1x24 jam sebelum melaporkan ke customer service.', 'category': 'Pengiriman', 'distance': np.float32(0.61870193)}
{'rank': 3, 'question': 'Kenapa nomor resi saya belum bisa dilacak di website kurir?', 'answer': "Nomor resi baru dapat dilacak setelah paket diserahterimakan ke pihak kurir, biasanya butuh waktu beberapa jam setelah status 'Dikirim'.", 'category': 'Pengiriman', 'distance': np.float32(0.634567)}


## 6. Mendesain Prompt dan Context (Persona, Task, Context, Format)

Susun prompt terstruktur (Persona, Task, Context, Format) yang menyisipkan hasil semantic search sebagai bagian Context, sebelum dikirim ke LLM.

In [23]:
def build_prompt(query, search_results):
    context = '\n\n'.join(
        f"FAQ: {r['question']}\nJawaban: {r['answer']}" for r in search_results
    )
    prompt = f'''Kamu adalah asisten customer service e-commerce yang ramah dan membantu.

Tugas: Jawab pertanyaan pengguna HANYA berdasarkan informasi FAQ berikut. Jika informasi tidak ditemukan dalam FAQ, katakan bahwa kamu belum memiliki informasi tersebut dan sarankan menghubungi customer service.

Konteks FAQ relevan:
{context}

Pertanyaan pengguna: {query}

Jawaban:'''
    return prompt

In [27]:
# def build_prompt(query, results):
#     context = "\n\n".join(
#         f"[Sumber {int(r['rank'])}]\n{r['retrieval_text']}" for _, r in results.iterrows()
#     )
#     return (
#         "## Persona\n"
#         "Kamu adalah asisten customer service e-commerce yang ramah, sopan, dan profesional.\n\n"
#         "## Task\n"
#         "Jawab pertanyaan pelanggan HANYA berdasarkan informasi pada bagian Context. "
#         "Jangan memakai pengetahuan di luar Context dan jangan mengarang. "
#         "Jika Context tidak memuat jawaban yang relevan, katakan dengan jelas bahwa informasi tersebut "
#         "belum tersedia di knowledge base dan sarankan pelanggan menghubungi tim customer service.\n\n"
#         f"## Context\n{context}\n\n"
#         "## Format\n"
#         "- Jawab dalam Bahasa Indonesia yang natural, maksimal 3-4 kalimat.\n"
#         "- Gunakan poin-poin hanya jika jawabannya berupa langkah-langkah.\n"
#         "- Jangan menyebut kata 'Context' atau 'Sumber' dalam jawaban.\n\n"
#         f"## Pertanyaan Pelanggan\n{query}\n\n"
#         "## Jawaban\n"
#     )


# demo = semantic_search("bagaimana cara melacak pesanan saya?")
# print(build_prompt("bagaimana cara melacak pesanan saya?", demo))

## 7. Mengintegrasikan LLM

In [41]:
def generate_answer(prompt):
    resp = with_retry(lambda: client.models.generate_content(
        model='gemini-3.1-flash-lite',  # sesuaikan versi model,
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.2),
    ))
    return resp.text.strip()


def rag_answer(query, top_k=TOP_K):
    results = semantic_search(query, top_k)        # query -> retrieval
    prompt = build_prompt(query, results)          # -> prompt
    answer = generate_answer(prompt)               # -> jawaban
    return results, answer

## 8. Menguji Relevansi Hasil Retrieval dan Response AI

Uji sistem dengan beberapa query berbeda, termasuk query yang menggunakan kata/frasa berbeda dari FAQ asli (sinonim/parafrase) untuk menilai apakah semantic search benar-benar menangkap makna, bukan hanya mencocokkan kata kunci.

_Uji minimal 5 query berbeda, termasuk:_
- _1-2 query yang jelas ada jawabannya di FAQ_
- _1-2 query dengan kata-kata berbeda dari FAQ asli (parafrase/sinonim)_
- _1 query yang TIDAK ada jawabannya di FAQ (uji bagaimana LLM merespons ketidaktahuannya)_

In [43]:
test_queries = [
    # TODO: isi minimal 5 query uji coba di sini
    "apakah bisa pesan gak ada akun",
    "berpa lama barang dikirim ?",
    "bagaimana pembatalan pesananan",
    "bagaimana melacak pengiriman",
    "apakah bisa bayar di tempat",
    "apakah bisa dikirim ke luar negeri ?"
]

for q in test_queries:
    print('Query:', q)
    answer, sources = generate_answer(q)
    print('Jawaban:', answer)
    print('Sumber teratas:', sources[0]['question'] if sources else '-')
    print('---')

Query: apakah bisa pesan gak ada akun


ValueError: too many values to unpack (expected 2)

### Catatan Analisis Relevansi

_Tuliskan pengamatanmu di sini: query mana yang berhasil di-retrieve dengan baik, query mana yang kurang relevan hasilnya, dan bagaimana LLM merespons ketika informasi tidak tersedia di knowledge base._

- query yang berhasil d retrieve dengan baik adalah pertanyaan yang sesuai dengan document yang ada dan walaupun katanya tidak sesuai keonteks atau ditambah non kata respon atau jawaban sesuai dengan yang ada di dokumen
- dan apabila pertanyaan tidak ada di knowledge base/document namun masih merespon dengan masuk akal sesuai dengan pertanyaan yang dimasukan